# Complete Locust Test Collection

This notebook collects all 24 Locust examples in learning order. Each code cell is the exact source of its corresponding `locustfile.py`. The canonical runnable files remain under `locust_tests/`.

Run an example from the repository root with:

```sh
locust -f locust_tests/locust_001/locustfile.py --host=https://httpbin.org
```

Only load-test systems you own or have explicit permission to test. Keep user counts low when using public demonstration APIs.

## Index

1. **locust_001** — First GET request
2. **locust_002** — Weighted tasks
3. **locust_003** — GET requests and query parameters
4. **locust_004** — POST requests
5. **locust_005** — Response validation
6. **locust_006** — PUT requests
7. **locust_007** — DELETE requests
8. **locust_008** — Direct weighted tasks
9. **locust_009** — Repeating ordered tasks
10. **locust_010** — Grouped random tasks
11. **locust_011** — A sequential user journey
12. **locust_012** — Nested task sets
13. **locust_013** — Constant concurrent-user load
14. **locust_014** — Constant task throughput
15. **locust_015** — Step load
16. **locust_016** — Spike load
17. **locust_017** — Fixed iteration pacing
18. **locust_018** — User lifecycle and state
19. **locust_019** — Response correlation
20. **locust_020** — Task tags and filtering
21. **locust_021** — Multiple user classes
22. **locust_022** — Automated thresholds
23. **locust_023** — Custom CLI arguments
24. **locust_024** — Faster HTTP client


## 01 — First GET request

Source: `locust_tests/locust_001/locustfile.py`

In [ ]:
from locust import HttpUser, between, task


class WebsiteUser(HttpUser):
    wait_time = between(1, 3)

    @task
    def get_data(self):
        self.client.get("/get")


## 02 — Weighted tasks

Source: `locust_tests/locust_002/locustfile.py`

In [ ]:
from locust import HttpUser, between, task


class WebsiteUser(HttpUser):
    wait_time = between(1, 3)

    @task(3)
    def get_data(self):
        self.client.get("/get")

    @task(1)
    def get_headers(self):
        self.client.get("/headers")


## 03 — GET requests and query parameters

Source: `locust_tests/locust_003/locustfile.py`

In [ ]:
import random

from locust import HttpUser, between, task


class WebsiteUser(HttpUser):
    wait_time = between(1, 3)

    @task
    def list_posts(self):
        self.client.get("/posts", name="/posts")

    @task
    def view_fixed_post(self):
        self.client.get("/posts/1", name="/posts/[id]")

    @task
    def view_random_post(self):
        post_id = random.randint(1, 100)
        self.client.get(f"/posts/{post_id}", name="/posts/[id]")

    @task
    def search_posts(self):
        user_id = random.randint(1, 10)
        self.client.get(
            "/posts",
            params={"userId": user_id},
            name="/posts?userId=[id]",
        )


## 04 — POST requests

Source: `locust_tests/locust_004/locustfile.py`

In [ ]:
import random

from locust import HttpUser, between, task


POST_TITLES = (
    "The Silence Between Stars",
    "Where the Wild Roads End",
    "The Last Letter at Midnight",
)

POST_BODIES = (
    "An astronomer discovers a message hidden between distant stars.",
    "Two strangers follow an abandoned road toward a town missing from every map.",
    "A midnight letter reveals a family's long-buried secret.",
)


class WebsiteUser(HttpUser):
    wait_time = between(1, 3)

    @task
    def create_post(self):
        self.client.post(
            "/posts",
            json={
                "title": random.choice(POST_TITLES),
                "body": random.choice(POST_BODIES),
                "userId": random.randint(1, 10),
            },
        )


## 05 — Response validation

Source: `locust_tests/locust_005/locustfile.py`

In [ ]:
import random

from locust import HttpUser, between, task


POST_TITLES = (
    "The Silence Between Stars",
    "Where the Wild Roads End",
    "The Last Letter at Midnight",
)

POST_BODIES = (
    "An astronomer discovers a message hidden between distant stars.",
    "Two strangers follow an abandoned road toward a town missing from every map.",
    "A midnight letter reveals a family's long-buried secret.",
)


class WebsiteUser(HttpUser):
    wait_time = between(1, 3)

    @task
    def create_post(self):
        with self.client.post(
            "/posts",
            json={
                "title": random.choice(POST_TITLES),
                "body": random.choice(POST_BODIES),
                "userId": random.randint(1, 10),
            },
            catch_response=True,
        ) as response:
            if response.status_code != 201:
                response.failure(f"Expected status 201, got {response.status_code}")
                return

            try:
                response_data = response.json()
            except ValueError:
                response.failure("Response body was not valid JSON")
                return

            if "id" not in response_data:
                response.failure("Response JSON did not contain an id")
            else:
                response.success()


## 06 — PUT requests

Source: `locust_tests/locust_006/locustfile.py`

In [ ]:
import random

from locust import HttpUser, between, task


POST_TITLES = (
    "The Silence Between Stars",
    "Where the Wild Roads End",
    "The Last Letter at Midnight",
)

POST_BODIES = (
    "An astronomer discovers a message hidden between distant stars.",
    "Two strangers follow an abandoned road toward a town missing from every map.",
    "A midnight letter reveals a family's long-buried secret.",
)


class WebsiteUser(HttpUser):
    wait_time = between(1, 3)

    @task
    def update_post(self):
        post_id = random.randint(1, 100)
        self.client.put(
            f"/posts/{post_id}",
            name="/posts/[id]",
            json={
                "id": post_id,
                "title": random.choice(POST_TITLES),
                "body": random.choice(POST_BODIES),
                "userId": random.randint(1, 10),
            },
        )


## 07 — DELETE requests

Source: `locust_tests/locust_007/locustfile.py`

In [ ]:
import random

from locust import HttpUser, between, task


class WebsiteUser(HttpUser):
    wait_time = between(1, 3)

    @task
    def delete_post(self):
        post_id = random.randint(1, 100)
        self.client.delete(f"/posts/{post_id}", name="/posts/[id]")


## 08 — Direct weighted tasks

Source: `locust_tests/locust_008/locustfile.py`

In [ ]:
import random

from locust import HttpUser, between, task


class WebsiteUser(HttpUser):
    wait_time = between(1, 3)

    @task(5)
    def view_posts(self):
        self.client.get("/posts", name="/posts")

    @task(4)
    def view_post(self):
        self.client.get("/posts/1", name="/posts/[id]")

    @task(3)
    def view_random_post(self):
        post_id = random.randint(1, 100)
        self.client.get(f"/posts/{post_id}", name="/posts/[id]")

    @task(2)
    def search_posts(self):
        user_id = random.randint(1, 10)
        self.client.get("/posts", params={"userId": user_id}, name="/posts?userId=[id]")


## 09 — Repeating ordered tasks

Source: `locust_tests/locust_009/locustfile.py`

In [ ]:
import random

from locust import HttpUser, SequentialTaskSet, between, task


class BrowseSequence(SequentialTaskSet):
    @task
    def view_posts(self):
        self.client.get("/posts", name="/posts")

    @task
    def view_post(self):
        self.client.get("/posts/1", name="/posts/[id]")

    @task
    def view_random_post(self):
        post_id = random.randint(1, 100)
        self.client.get(f"/posts/{post_id}", name="/posts/[id]")

    @task
    def search_posts(self):
        user_id = random.randint(1, 10)
        self.client.get("/posts", params={"userId": user_id}, name="/posts?userId=[id]")


class WebsiteUser(HttpUser):
    wait_time = between(1, 3)
    tasks = [BrowseSequence]


## 10 — Grouped random tasks

Source: `locust_tests/locust_010/locustfile.py`

In [ ]:
import random

from locust import HttpUser, TaskSet, between, task


class ReadPosts(TaskSet):
    @task(3)
    def list_posts(self):
        self.client.get("/posts", name="/posts")

    @task(2)
    def view_post(self):
        post_id = random.randint(1, 100)
        self.client.get(f"/posts/{post_id}", name="/posts/[id]")

    @task(1)
    def leave_read_section(self):
        self.interrupt()


class WritePosts(TaskSet):
    @task(2)
    def create_post(self):
        self.client.post(
            "/posts",
            json={
                "title": "A Locust test post",
                "body": "Created during a load test",
                "userId": random.randint(1, 10),
            },
        )

    @task(1)
    def delete_post(self):
        post_id = random.randint(1, 100)
        self.client.delete(f"/posts/{post_id}", name="/posts/[id]")

    @task(1)
    def leave_write_section(self):
        self.interrupt()


class WebsiteUser(HttpUser):
    wait_time = between(1, 3)
    tasks = {ReadPosts: 3, WritePosts: 1}


## 11 — A sequential user journey

Source: `locust_tests/locust_011/locustfile.py`

In [ ]:
import random

from locust import HttpUser, SequentialTaskSet, between, task


class PostJourney(SequentialTaskSet):
    def on_start(self):
        self.post_id = random.randint(1, 100)

    @task
    def list_posts(self):
        self.client.get("/posts", name="/posts")

    @task
    def view_post(self):
        self.client.get(f"/posts/{self.post_id}", name="/posts/[id]")

    @task
    def view_comments(self):
        self.client.get(
            f"/posts/{self.post_id}/comments",
            name="/posts/[id]/comments",
        )

    @task
    def finish_journey(self):
        self.interrupt()


class WebsiteUser(HttpUser):
    wait_time = between(1, 3)
    tasks = [PostJourney]


## 12 — Nested task sets

Source: `locust_tests/locust_012/locustfile.py`

In [ ]:
import random

from locust import HttpUser, TaskSet, between, task


class CommentTasks(TaskSet):
    @task(3)
    def list_comments(self):
        self.client.get("/comments", name="/comments")

    @task(1)
    def view_comment(self):
        comment_id = random.randint(1, 100)
        self.client.get(f"/comments/{comment_id}", name="/comments/[id]")

    @task(1)
    def leave_comments(self):
        self.interrupt()


class PostTasks(TaskSet):
    tasks = {CommentTasks: 2}

    @task(3)
    def list_posts(self):
        self.client.get("/posts", name="/posts")

    @task(2)
    def view_post(self):
        post_id = random.randint(1, 100)
        self.client.get(f"/posts/{post_id}", name="/posts/[id]")

    @task(1)
    def leave_posts(self):
        self.interrupt()


class WebsiteUser(HttpUser):
    wait_time = between(1, 3)
    tasks = [PostTasks]


## 13 — Constant concurrent-user load

Source: `locust_tests/locust_013/locustfile.py`

In [ ]:
from locust import HttpUser, LoadTestShape, between, task


class WebsiteUser(HttpUser):
    wait_time = between(1, 3)

    @task
    def list_posts(self):
        self.client.get("/posts")


class ConstantLoadShape(LoadTestShape):
    users = 10
    spawn_rate = 2
    duration = 60

    def tick(self):
        if self.get_run_time() < self.duration:
            return self.users, self.spawn_rate

        return None


## 14 — Constant task throughput

Source: `locust_tests/locust_014/locustfile.py`

In [ ]:
from locust import HttpUser, constant_throughput, task


class WebsiteUser(HttpUser):
    wait_time = constant_throughput(1)

    @task
    def list_posts(self):
        self.client.get("/posts")


## 15 — Step load

Source: `locust_tests/locust_015/locustfile.py`

In [ ]:
from locust import HttpUser, LoadTestShape, between, task


class WebsiteUser(HttpUser):
    wait_time = between(1, 3)

    @task
    def list_posts(self):
        self.client.get("/posts")


class StepLoadShape(LoadTestShape):
    step_duration = 30
    users_per_step = 5
    spawn_rate = 2
    duration = 120

    def tick(self):
        run_time = self.get_run_time()

        if run_time >= self.duration:
            return None

        step = int(run_time // self.step_duration) + 1
        return step * self.users_per_step, self.spawn_rate


## 16 — Spike load

Source: `locust_tests/locust_016/locustfile.py`

In [ ]:
from locust import HttpUser, LoadTestShape, between, task


class WebsiteUser(HttpUser):
    wait_time = between(1, 3)

    @task
    def list_posts(self):
        self.client.get("/posts")


class SpikeLoadShape(LoadTestShape):
    stages = (
        {"duration": 30, "users": 5, "spawn_rate": 2},
        {"duration": 45, "users": 30, "spawn_rate": 15},
        {"duration": 75, "users": 5, "spawn_rate": 15},
    )

    def tick(self):
        run_time = self.get_run_time()

        for stage in self.stages:
            if run_time < stage["duration"]:
                return stage["users"], stage["spawn_rate"]

        return None


## 17 — Fixed iteration pacing

Source: `locust_tests/locust_017/locustfile.py`

In [ ]:
from locust import HttpUser, constant_pacing, task


class WebsiteUser(HttpUser):
    wait_time = constant_pacing(2)

    @task
    def list_posts(self):
        self.client.get("/posts")


## 18 — User lifecycle and state

Source: `locust_tests/locust_018/locustfile.py`

In [ ]:
import random

from locust import HttpUser, between, task


class WebsiteUser(HttpUser):
    wait_time = between(1, 3)

    def on_start(self):
        self.user_id = random.randint(1, 10)
        self.client.get(f"/users/{self.user_id}", name="/users/[id]")

    @task
    def view_my_posts(self):
        self.client.get(
            "/posts",
            params={"userId": self.user_id},
            name="/posts?userId=[id]",
        )

    def on_stop(self):
        self.user_id = None


## 19 — Response correlation

Source: `locust_tests/locust_019/locustfile.py`

In [ ]:
from locust import HttpUser, between, task


class WebsiteUser(HttpUser):
    wait_time = between(1, 3)

    @task
    def view_post_author(self):
        with self.client.get(
            "/posts/1",
            name="/posts/[id]",
            catch_response=True,
        ) as response:
            try:
                user_id = response.json()["userId"]
            except (KeyError, ValueError):
                response.failure("Post response did not contain a userId")
                return

        self.client.get(f"/users/{user_id}", name="/users/[id]")


## 20 — Task tags and filtering

Source: `locust_tests/locust_020/locustfile.py`

In [ ]:
import random

from locust import HttpUser, between, tag, task


class WebsiteUser(HttpUser):
    wait_time = between(1, 3)

    @tag("read")
    @task(3)
    def list_posts(self):
        self.client.get("/posts")

    @tag("read", "detail")
    @task(2)
    def view_post(self):
        post_id = random.randint(1, 100)
        self.client.get(f"/posts/{post_id}", name="/posts/[id]")

    @tag("write")
    @task(1)
    def create_post(self):
        self.client.post(
            "/posts",
            json={"title": "Tagged task", "body": "Example", "userId": 1},
        )


## 21 — Multiple user classes

Source: `locust_tests/locust_021/locustfile.py`

In [ ]:
import random

from locust import HttpUser, between, constant, task


class BrowserUser(HttpUser):
    weight = 3
    wait_time = between(1, 3)

    @task
    def browse_posts(self):
        post_id = random.randint(1, 100)
        self.client.get(f"/posts/{post_id}", name="/posts/[id]")


class AuthorUser(HttpUser):
    weight = 1
    wait_time = between(2, 5)

    @task
    def create_post(self):
        self.client.post(
            "/posts",
            json={"title": "New post", "body": "Example", "userId": 1},
        )


class MonitorUser(HttpUser):
    fixed_count = 1
    wait_time = constant(10)

    @task
    def check_service(self):
        self.client.get("/posts/1", name="/posts/[id]")


## 22 — Automated thresholds

Source: `locust_tests/locust_022/locustfile.py`

In [ ]:
import logging

from locust import HttpUser, between, events, task


MAX_FAILURE_RATIO = 0.01
MAX_AVERAGE_RESPONSE_TIME = 1_000
MAX_P95_RESPONSE_TIME = 2_000


@events.quitting.add_listener
def enforce_performance_thresholds(environment, **_kwargs):
    stats = environment.stats.total

    if stats.fail_ratio > MAX_FAILURE_RATIO:
        logging.error("Failure ratio exceeded 1%%")
        environment.process_exit_code = 1
    elif stats.avg_response_time > MAX_AVERAGE_RESPONSE_TIME:
        logging.error("Average response time exceeded 1000 ms")
        environment.process_exit_code = 1
    elif stats.get_response_time_percentile(0.95) > MAX_P95_RESPONSE_TIME:
        logging.error("p95 response time exceeded 2000 ms")
        environment.process_exit_code = 1
    else:
        environment.process_exit_code = 0


class WebsiteUser(HttpUser):
    wait_time = between(1, 3)

    @task
    def list_posts(self):
        self.client.get("/posts")


## 23 — Custom CLI arguments

Source: `locust_tests/locust_023/locustfile.py`

In [ ]:
from locust import HttpUser, between, events, task


@events.init_command_line_parser.add_listener
def add_custom_arguments(parser):
    parser.add_argument(
        "--post-limit",
        type=int,
        default=10,
        env_var="LOCUST_POST_LIMIT",
        help="Number of posts requested by each task",
    )


class WebsiteUser(HttpUser):
    wait_time = between(1, 3)

    @task
    def list_posts(self):
        post_limit = self.environment.parsed_options.post_limit
        self.client.get(
            "/posts",
            params={"_limit": post_limit},
            name="/posts?_limit=[value]",
        )


## 24 — Faster HTTP client

Source: `locust_tests/locust_024/locustfile.py`

In [ ]:
from locust import FastHttpUser, between, task


class WebsiteUser(FastHttpUser):
    wait_time = between(1, 3)

    @task
    def list_posts(self):
        self.client.get("/posts")
